<div class="align-center">
<a href="https://oumi.ai/"><img src="https://oumi.ai/docs/en/latest/_static/logo/header_logo.png" height="200"></a>

[![Documentation](https://img.shields.io/badge/Documentation-latest-blue.svg)](https://oumi.ai/docs/en/latest/index.html)
[![Discord](https://img.shields.io/badge/Discord-Oumi-blue?logo=discord)](https://discord.gg/oumi)
[![GitHub Stars](https://img.shields.io/github/stars/oumi-ai/oumi?style=social)](https://github.com/oumi-ai/oumi)
[![GitHub Issues](https://img.shields.io/github/issues/oumi-ai/oumi)](https://github.com/oumi-ai/oumi/issues)
[![License](https://img.shields.io/badge/License-Apache%202.0-green.svg)](https://github.com/oumi-ai/oumi/blob/main/LICENSE)
</div>

# AIDE Agentic Code Optimization

Welcome to Oumi! In this notebook, we'll use [**AIDE (AI-Driven Exploration)**](https://github.com/WecoAI/aideml) to autonomously optimize ML training code using LLM-powered tree search.

Unlike traditional hyperparameter tuning (`oumi tune` / Optuna), which searches over a predefined parameter space, AIDE operates in **code space** — it uses an LLM to actually *write and rewrite Python code*, execute it, evaluate the results, and iteratively improve. This means it can discover things that parameter tuning never could: novel optimizer configurations, data preprocessing strategies, reward function designs, and entirely different training approaches.

We'll cover the following topics:
1. Setup — Install, LLM access (local or API), tutorial directory
2. How AIDE Works — The Tree Search Algorithm
3. Building and Running an Optimization
4. Using YAML + CLI
5. Optimization Surfaces
6. AIDE vs Optuna — When to Use Which
7. Tips for Best Results
8. References & Credits

## 1. Setup

### Install

In [ ]:
%pip install uv
!cd .. && uv pip install -e ".[aide]"

# For production: !uv pip install "oumi[aide]"

Note: you may need to restart the kernel to use updated packages.
Using Python 3.11.15 environment at: /opt/miniconda3/envs/oumi
Resolved 339 packages in 836ms                                       
   Building oumi @ file:///Users/siatras/Documents/GitHub/oumi         
   Building oumi @ file:///Users/siatras/Documents/GitHub/oumi 
      Built oumi @ file:///Users/siatras/Documents/GitHub/oumi 
Prepared 1 package in 400ms                                              
Uninstalled 1 package in 0.66ms
Installed 1 package in 1ms823ec57.d20260315 (from file:///Us
 ~ oumi==0.8.dev55+g1c823ec57.d20260315 (from file:///Users/siatras/Documents/GitHub/oumi)


### LLM Access

AIDE needs an LLM to generate and evaluate code. You have two options:

#### Option A: Local model (recommended, free)

Serve a coding-capable model locally using vLLM or Ollama. Any OpenAI-compatible server works. Use a strong coding model for best results.

```bash
# Using vLLM:
vllm serve Qwen/Qwen3-Coder-Next --port 8000

# Or using Ollama:
ollama serve && ollama run qwen3-coder-next
```

Then set the environment variables below.

#### Option B: Cloud API

If you don't have a GPU for local serving, use a cloud LLM API instead.
Cost is typically ~$0.01–0.10 per search step.

In [ ]:
import os

# === Option A: Local model server (free, no API key needed) ===
# Uncomment these if you have a local vLLM/Ollama server running:
# os.environ["OPENAI_BASE_URL"] = "http://localhost:8000/v1"
# os.environ["OPENAI_API_KEY"] = "dummy"

# === Option B: Cloud API (uncomment ONE) ===
# os.environ["OPENAI_API_KEY"] = ""      # For gpt-5-mini, etc.
# os.environ["ANTHROPIC_API_KEY"] = ""   # For claude-sonnet-4, etc.
# os.environ["GEMINI_API_KEY"] = ""      # For gemini-2.5-flash, etc.

### Tutorial Directory

In [5]:
from pathlib import Path

tutorial_dir = "aide_tutorial"
Path(tutorial_dir).mkdir(parents=True, exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

### Detect LLM Provider

AIDE's `code_llm` model name must match your provider. This cell auto-detects which one you configured and sets the right models.

In [6]:
# Auto-detect which LLM provider is available.
has_local = bool(os.environ.get("OPENAI_BASE_URL"))
has_openai = bool(os.environ.get("OPENAI_API_KEY")) and not has_local
has_anthropic = bool(os.environ.get("ANTHROPIC_API_KEY"))
has_gemini = bool(os.environ.get("GEMINI_API_KEY"))

if has_local:
    # Local server — change to match your served model name.
    CODE_MODEL = "Qwen/Qwen3-Coder-Next"
    FEEDBACK_MODEL = CODE_MODEL
    provider = "Local server"
elif has_openai:
    CODE_MODEL = "o4-mini"
    FEEDBACK_MODEL = "gpt-5-mini"
    provider = "OpenAI"
elif has_anthropic:
    CODE_MODEL = "claude-sonnet-4-6"
    FEEDBACK_MODEL = "claude-sonnet-4-6"
    provider = "Anthropic"
elif has_gemini:
    CODE_MODEL = "gemini-2.5-flash"
    FEEDBACK_MODEL = "gemini-2.5-flash"
    provider = "Gemini"
else:
    CODE_MODEL = "o4-mini"
    FEEDBACK_MODEL = "gpt-5-mini"
    provider = None

if provider:
    print(f"{provider} detected.")
    print(f"  Code LLM:     {CODE_MODEL}")
    print(f"  Feedback LLM: {FEEDBACK_MODEL}")
else:
    print("No LLM provider configured.")
    print("Set OPENAI_BASE_URL (local) or an API key (OPENAI/ANTHROPIC/GEMINI).")
    print("The cells below will show example output instead.")

Anthropic detected.
  Code LLM:     claude-sonnet-4-6
  Feedback LLM: claude-sonnet-4-6


## 2. How AIDE Works — The Tree Search Algorithm

AIDE frames ML engineering as a **code optimization problem** solved via tree search:

1. **Search Policy** decides the next action:
   - If fewer than `num_drafts` solutions exist → **Draft** a new solution from scratch
   - With probability `debug_prob`, if there are buggy nodes → **Debug** by fixing errors
   - Otherwise → **Improve** the best-performing solution

2. **Code Generation** — The `code_llm` writes a Python script that trains a model and prints a metric

3. **Execution** — The script runs in an isolated subprocess with timeout and output capture

4. **Review** — The `feedback_llm` analyzes the output: did it work? what's the metric?

5. **Tree Update** — The solution joins the tree. Best solutions get improved, buggy ones get debugged.

```
                    ┌─── draft_1 (buggy) ──── debug_1 (fixed, metric=0.82)
                    │
    root ───────────┼─── draft_2 (metric=0.71) ──── improve_1 (metric=0.65) ← best!
                    │
                    └─── draft_3 (metric=0.78) ──── improve_2 (metric=0.73)
```

## 3. Building and Running an Optimization

Let's optimize training hyperparameters for SmolLM 135M on Alpaca. We specify the model, data, a natural language goal, which config fields AIDE can modify, and the search settings.

In [ ]:
from oumi.core.configs import (  # type: ignore
    AideConfig,
    AideParams,
    AideSearchParams,
    DataParams,
    DatasetParams,
    DatasetSplitParams,
    ModelParams,
)
from oumi.core.configs.params.aide_params import (  # type: ignore
    AideExecParams,
    AideLLMParams,
    AideOptimizationSurface,
)

config = AideConfig(
    model=ModelParams(
        model_name="HuggingFaceTB/SmolLM2-135M-Instruct",
        torch_dtype_str="bfloat16",
        trust_remote_code=True,
    ),
    data=DataParams(
        train=DatasetSplitParams(
            datasets=[
                DatasetParams(
                    dataset_name="yahma/alpaca-cleaned",
                    split="train[:90%]",
                ),
            ],
        ),
    ),
    goal=(
        "Optimize training hyperparameters for SmolLM 135M on Alpaca "
        "to minimize eval_loss. Focus on learning rate, optimizer, "
        "and warmup schedule. Keep each trial under 10 minutes."
    ),
    base_training_config="configs/recipes/smollm/sft/135m/train.yaml",
    mutable_config_paths=[
        "training.learning_rate",
        "training.optimizer",
        "training.warmup_ratio",
    ],
    aide=AideParams(
        steps=5,
        surface=AideOptimizationSurface.CONFIG_SEARCH,
        target_metric="eval_loss",
        target_direction="minimize",
        output_dir=f"{tutorial_dir}/output",
        workspace_dir=f"{tutorial_dir}/workspace",
        code_llm=AideLLMParams(model=CODE_MODEL, temperature=0.5),
        feedback_llm=AideLLMParams(model=FEEDBACK_MODEL, temperature=0.5),
        search=AideSearchParams(num_drafts=2, debug_prob=0.5),
        execution=AideExecParams(timeout=600),
    ),
)

print("AIDE Configuration:")
print(f"  Surface:    {config.aide.surface.value}")
print(f"  Steps:      {config.aide.steps}")
print(f"  Metric:     {config.aide.target_metric}")
print(f"  Code LLM:   {config.aide.code_llm.model}")
print(f"  Timeout:    {config.aide.execution.timeout}s")

In [ ]:
from oumi.aide import aide as run_aide  # type: ignore

if provider:
    result = run_aide(config, verbose=True)

    print(f"\nBest metric ({config.aide.target_metric}): {result.best_metric}")
    print(f"Total steps: {result.total_steps}")
    print(f"Good: {result.good_solutions}, Buggy: {result.buggy_solutions}")
    print(f"Best solution: {result.best_solution_path}")
    print(f"Journal: {result.journal_path}")
else:
    print("[Skipped \u2014 no LLM provider configured]")

[2026-03-15 13:02:16,632][oumi][rank0][pid:84346][MainThread][INFO]][torch_utils.py:80] Torch version: 2.10.0. NumPy version: 2.3.5
[2026-03-15 13:02:16,632][oumi][rank0][pid:84346][MainThread][INFO]][torch_utils.py:82] CUDA is not available!
[2026-03-15 13:02:16,635][oumi][rank0][pid:84346][MainThread][INFO]][aide.py:82] Oumi version: 0.8.dev55+g1c823ec57.d20260315
[2026-03-15 13:02:16,657][oumi][rank0][pid:84346][MainThread][INFO]][aide.py:84] Git revision hash: 1c823ec57589e6e9ff9d6787bb5564c7a1f25df6
[2026-03-15 13:02:16,684][oumi][rank0][pid:84346][MainThread][INFO]][aide.py:85] Git tag: None
[2026-03-15 13:02:16,687][oumi][rank0][pid:84346][MainThread][INFO]][aide.py:129] AideConfig:
AideConfig(data=DataParams(train=DatasetSplitParams(datasets=[DatasetParams(dataset_name='yahma/alpaca-cleaned',
                                                                            dataset_path=None,
                                                                            subset=None,
    

## 4. Using YAML + CLI

The same config can be written as YAML for reproducible experiments:

In [ ]:
%%writefile $tutorial_dir/aide_config.yaml

model:
  model_name: "HuggingFaceTB/SmolLM2-135M-Instruct"
  torch_dtype_str: "bfloat16"
  trust_remote_code: true

data:
  train:
    datasets:
      - dataset_name: "yahma/alpaca-cleaned"
        split: "train[:90%]"

goal: >
  Optimize training hyperparameters for SmolLM 135M on Alpaca
  to minimize eval_loss.

base_training_config: "configs/recipes/smollm/sft/135m/train.yaml"

mutable_config_paths:
  - "training.learning_rate"
  - "training.optimizer"
  - "training.warmup_ratio"

aide:
  steps: 5
  surface: CONFIG_SEARCH
  target_metric: "eval_loss"
  target_direction: "minimize"
  code_llm:
    model: "o4-mini"                        # Change to match your provider
  feedback_llm:
    model: "gpt-5-mini"                    # Change to match your provider
  search:
    num_drafts: 2
  execution:
    timeout: 600

Overwriting aide_tutorial/aide_config.yaml


Run via CLI:

```bash
# From YAML file
oumi aide -c aide_tutorial/aide_config.yaml

# Built-in recipe alias
oumi aide -c smollm-135m

# Override parameters
oumi aide -c smollm-135m --aide.steps=20
```

In [ ]:
!oumi aide --help


   ____  _    _ __  __ _____
  / __ \| |  | |  \/  |_   _|
 | |  | | |  | | \  / | | |
 | |  | | |  | | |\/| | | |
 | |__| | |__| | |  | |_| |_
  \____/ \____/|_|  |_|_____|

                                                                                
 Usage: oumi aide [OPTIONS]                                                     
                                                                                
 Run AI-driven agentic code optimization.                                       
                                                                                
 Example configs:                                                               
 • smollm-135m                                                                  
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ *  --config     -c        TEXT                     Path or config name (e.g. │
│             

## 5. Optimization Surfaces

AIDE supports multiple surfaces, controlled by `aide.surface` in the config:

### CONFIG_SEARCH (default)
Generates scripts that modify Oumi `TrainingConfig` parameters. Use `mutable_config_paths` to constrain the search.

**Best for:** Learning rates, optimizers, schedulers, LoRA/PEFT settings.

> **Tip:** To search LoRA hyperparameters, add `"peft.lora_r"`, `"peft.lora_alpha"`, and `"peft.lora_dropout"` to `mutable_config_paths`.

### REWARD_FUNCTION
Generates reward functions for GRPO/RLHF with signature `def reward_fn(completions, prompts, **kwargs) -> list[float]`.

**Best for:** Math reasoning rewards, code scoring, safety trade-offs. See the [AIDE Reward Function Design](Oumi%20-%20AIDE%20Reward%20Function%20Design.ipynb) notebook.

### EVAL_FUNCTION
Generates custom evaluation functions using `@register_evaluation_function`.

**Best for:** Domain-specific benchmarks, custom metrics. See the [AIDE Custom Evaluation](Oumi%20-%20AIDE%20Custom%20Evaluation.ipynb) notebook.

### FULL_PIPELINE
Generates complete training scripts. No constraints — maximum flexibility.

**Best for:** Open-ended exploration when you want AIDE to try fundamentally different approaches.

To use a different surface, just change the config:
```yaml
aide:
  surface: REWARD_FUNCTION       # or EVAL_FUNCTION, FULL_PIPELINE
  target_metric: "reward_accuracy"
  target_direction: "maximize"
```

## 6. AIDE vs Optuna — When to Use Which

Oumi provides two optimization approaches. `oumi tune` currently uses Optuna for hyperparameter search.

| Feature | `oumi tune` (Optuna) | `oumi aide` (AIDE) |
|---------|---------------------|--------------------|
| **Search space** | Predefined numeric/categorical | Any Python code |
| **Algorithm** | Bayesian optimization (TPE) | LLM-powered tree search |
| **Requires** | Nothing extra | LLM (local or API) |
| **Best for** | Known parameter ranges | Open-ended optimization |
| **Cost** | Compute only | Compute + LLM (free with local model) |

**Rules of thumb:**
- **`oumi tune`** when you know which 3–5 parameters to search
- **`oumi aide`** when you want code-level changes or novel approaches
- **Both together:** AIDE to discover approaches, then Optuna to fine-tune specific parameters

**See also:** [AIDE Reward Function Design](Oumi%20-%20AIDE%20Reward%20Function%20Design.ipynb) | [AIDE Custom Evaluation](Oumi%20-%20AIDE%20Custom%20Evaluation.ipynb)

## 7. Tips for Best Results

- **Write detailed goals.** "Minimize eval_loss by exploring learning rates 1e-5 to 1e-2 and cosine vs linear schedulers" is better than "optimize the model".
- **Start small.** `steps=5` first, then scale to 20+ once you verify the agent understands the task.
- **Constrain the search.** `mutable_config_paths` prevents irrelevant changes.
- **Check the journal.** The `journal_path` file contains every solution, its code, metric, and whether it was buggy.
- **Try different LLMs.** `o4-mini` is fast, `claude-sonnet-4` may be more creative, a local 32B model is free.

## Summary

In this tutorial, you learned how to:

1. ✅ Set up AIDE with a local model server or cloud API
2. ✅ Understand the tree search algorithm (draft/debug/improve)
3. ✅ Configure and run AIDE optimization programmatically and via YAML/CLI
4. ✅ Choose between the four optimization surfaces
5. ✅ Decide when to use AIDE vs Optuna tuning

## 8. References & Credits

### AIDE — AI-Driven Exploration

AIDE is developed by [**Weco AI**](https://www.weco.ai/) and implements the tree-search algorithm from:

> **AIDE: AI-Driven Exploration in the Space of Code** — Jiang, Wu et al. [arXiv:2502.13138](https://arxiv.org/abs/2502.13138) (2025)

- **Weco AI:** [https://www.weco.ai/](https://www.weco.ai/)
- **AIDE ML Open Source:** [https://github.com/WecoAI/aideml](https://github.com/WecoAI/aideml)

### Oumi

- [https://oumi.ai/](https://oumi.ai/) | [GitHub](https://github.com/oumi-ai/oumi) | [Docs](https://oumi.ai/docs/)

# 🧭 What's Next?

Congrats on finishing this notebook! Feel free to check out our other [notebooks](https://github.com/oumi-ai/oumi/tree/main/notebooks) in the [Oumi GitHub](https://github.com/oumi-ai/oumi), and give us a star! You can also join the Oumi community over on [Discord](https://discord.gg/oumi).

- **[AIDE Reward Function Design](Oumi%20-%20AIDE%20Reward%20Function%20Design.ipynb)** — REWARD_FUNCTION surface: auto-design reward functions for GRPO
- **[AIDE Custom Evaluation](Oumi%20-%20AIDE%20Custom%20Evaluation.ipynb)** — EVAL_FUNCTION surface: auto-generate evaluation functions
- **[Finetuning Tutorial](Oumi%20-%20Finetuning%20Tutorial.ipynb)** — LoRA-tune models with Oumi
- **[GRPO Training](Oumi%20-%20Train%20a%20Letter%20Counting%20Model%20using%20GRPO.ipynb)** — Train with reinforcement learning

Happy optimizing!